In [6]:
import pandas as pd
import numpy as np
from prophet import Prophet

train_df = pd.read_csv("data/train.csv", parse_dates=["Date"])
store = pd.read_csv("data/store.csv")

df = train_df.merge(store, on="Store", how="left")

df = df[df["Store"] == 1].copy()
df = df.sort_values("Date")

df = df[["Date", "Sales", "Promo"]]

df = df.rename(columns={
    "Date": "ds",
    "Sales": "y",
    "Promo": "promo"
})

df = df.reset_index(drop=True)
split_index = int(len(df) * 0.8)

train = df.iloc[:split_index]
test = df.iloc[split_index:]

holidays = pd.DataFrame({
    "holiday": "generic_holiday",
    "ds": pd.to_datetime([
        "2013-12-25", "2014-12-25", "2015-12-25",
        "2013-01-01", "2014-01-01", "2015-01-01"
    ]),
    "lower_window": 0,
    "upper_window": 1
})

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    holidays=holidays
)

model.add_regressor("promo")
model.fit(train)

future = test[["ds", "promo"]].copy()
forecast = model.predict(future)

y_pred = forecast["yhat"].reset_index(drop=True)
y_test = test["y"].reset_index(drop=True)

mae = np.mean(np.abs(y_test - y_pred))
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))

y_test_np = y_test.values
y_pred_np = y_pred.values

nonzero = y_test_np != 0

mape = np.mean(np.abs(
    (y_test_np[nonzero] - y_pred_np[nonzero]) / y_test_np[nonzero]
)) * 100

print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

C:\Users\hp\AppData\Local\Temp\ipykernel_22436\3628898321.py:5: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("data/train.csv", parse_dates=["Date"])
21:42:45 - cmdstanpy - INFO - Chain [1] start processing
21:42:45 - cmdstanpy - INFO - Chain [1] done processing


MAE: 558.6345486864874
RMSE: 948.955430147648
MAPE: 10.34073895524741
